<a href="https://colab.research.google.com/github/atharv-0509/triage/blob/claude%2Fstarting-from-beginning-joh2ym/notebooks/phase2_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MetaPilot Phase 2: QLoRA fine-tuning of the LLM selector (Colab, Kaggle or any GPU)

Fine-tunes **Qwen2.5-1.5B-Instruct** with 4-bit QLoRA to rank the pipeline portfolio from a text profile of a
dataset, and writes its rankings so `metapilot evaluate` can score them next to the Phase 1 meta-models.

Protocol (same as Phase 1):
* one adapter per **outer fold** and prompt kind, trained only on tasks outside that fold;
* each adapter predicts only its own held-out fold, so every task is predicted by a model that never saw it;
* three variants: `base_numeric` (no fine-tuning), `ft_numeric` (Prompt A, same information as LightGBM) and
  `ft_semantic` (Prompt B, plus column names and description).

Nothing here falls back to a default ranking: every check below shows **coverage**, the share of pipelines the
model actually named, and how many **distinct rankings** it produced. Coverage near 0 means the model is not
producing rankings; very few distinct rankings means it learned one fixed order instead of reading the profile.

**Colab:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`, and allow Google Drive access.
Work is saved to Drive after every fold; after a disconnect, run all again and finished folds are skipped.

**Kaggle:** in the notebook's settings choose `Accelerator: GPU T4 x2` (one GPU is used) and turn
`Internet` on (needs a phone-verified account). Use `Save Version > Save & Run All (Commit)` so the run continues
in the background (up to 12 h) and `/kaggle/working` is kept as the version's output. To resume an unfinished
run, add that version's output as an input (`Add Input > Your Work`); finished folds are copied in and skipped.

In [ ]:
# 1. Settings
REPO = "https://github.com/atharv-0509/triage"
BRANCH = "claude/starting-from-beginning-joh2ym"
# Full OpenML study (meta-dataset shipped in the repo under metadata/full).
# For the pilot instead: CONFIG = "configs/pilot.yaml", META = "data/pilot", RUN_NAME = "pilot", EPOCHS = 3.
CONFIG = "configs/full.yaml"
META = "metadata/full"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
EPOCHS = 1                        # ~1,050 training tasks per fold; 1 epoch keeps the run to ~5-6 h on a T4
KINDS = ["numeric", "semantic"]
RUN_NAME = "full"                 # Drive subfolder
SHUFFLE_CANDIDATES = True         # list candidates in a different random order per task (see README)
REQUIRE_GPU = True

# Shuffled runs get their own folder and prediction names, so they never mix with unshuffled ones.
SUFFIX = "_shuf" if SHUFFLE_CANDIDATES else ""

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # one GPU (Kaggle's "T4 x2" would otherwise split the model)

In [ ]:
# 2. Platform, code and install (bitsandbytes is needed for 4-bit loading on the GPU)
import glob, shutil, subprocess
try:
    import google.colab  # noqa: F401
    PLATFORM = "colab"
except ImportError:
    PLATFORM = "kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "local"

if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = f"/content/drive/MyDrive/metapilot_phase2/{RUN_NAME}{SUFFIX}"
    CODE_DIR = "/content/triage"
elif PLATFORM == "kaggle":
    WORK_DIR = f"/kaggle/working/metapilot_phase2/{RUN_NAME}{SUFFIX}"
    CODE_DIR = os.path.expanduser("~/triage")
    previous = [p for p in glob.glob(f"/kaggle/input/**/metapilot_phase2/{RUN_NAME}{SUFFIX}", recursive=True)]
    if previous and not os.path.exists(WORK_DIR):
        shutil.copytree(previous[0], WORK_DIR)
        print("resuming from", previous[0])
else:  # any machine with a GPU, e.g. inside a clone of the repo
    CODE_DIR = os.getcwd() if os.path.isdir("metapilot") else os.path.expanduser("~/triage")
    WORK_DIR = os.path.join(CODE_DIR, "phase2_runs", RUN_NAME + SUFFIX)
os.makedirs(WORK_DIR, exist_ok=True)
print("platform:", PLATFORM, "| work dir:", WORK_DIR)

if not os.path.isdir(CODE_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO, CODE_DIR], check=True)
%cd {CODE_DIR}
if PLATFORM != "local":
    !git pull -q origin {BRANCH}
    !pip install -q -e ".[llm]" bitsandbytes

import torch
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU: choose a GPU runtime/accelerator, then run all again"
    print(torch.cuda.get_device_name(0))
from metapilot.llm_train import native_bf16
print("precision:", "bf16" if native_bf16() else "fp16")

In [ ]:
# 3. Meta-dataset and prompts.
# Pilot: built here. The full OpenML study ships in the repo: CONFIG = "configs/full.yaml",
# META = "metadata/full", RUN_NAME = "full". Any other meta-dataset: point META at its folder.
if not os.path.exists(f"{META}/performance.csv") and META.startswith("metadata/"):
    !git pull -q origin {BRANCH}
if not os.path.exists(f"{META}/performance.csv"):
    if META.startswith("data/"):
        !python -m metapilot build --config {CONFIG} --out {META}
        if _exit_code: raise RuntimeError("metapilot build failed (see the error above); stopping")
    else:
        raise FileNotFoundError(f"{META}/performance.csv not found: for metadata/full the OpenML build has not been "
                                f"pushed yet; otherwise copy the meta-dataset there first")

!python -m metapilot prompts --config {CONFIG} --meta {META} {"--shuffle-candidates" if SHUFFLE_CANDIDATES else ""}
if _exit_code: raise RuntimeError("metapilot prompts failed (see the error above); stopping")
PROMPTS = f"{META}/llm/prompts.jsonl"

import json
from metapilot import llm
rows = llm.load_jsonl(PROMPTS)
folds = sorted({r["fold"] for r in rows})
PIPELINES = rows[0]["target"].split(", ")
print(len(rows), "tasks,", len(PIPELINES), "pipelines, folds", folds)
print(rows[0]["numeric"]); print("TARGET:", rows[0]["target"])
print("candidates shuffled:", rows[0]["candidates_shuffled"])

In [ ]:
# Helper: coverage check on a predictions file (share of pipelines the model named itself)
import time
def check(path, show=2):
    preds = llm.load_jsonl(path) if os.path.exists(path) else []
    if not preds:
        print("no predictions yet:", path); return
    cov = [len(llm.parse_ranking(p["output"], PIPELINES)) / len(PIPELINES) for p in preds]
    full = sum(c == 1 for c in cov)
    distinct = len({tuple(llm.parse_ranking(p["output"], PIPELINES)) for p in preds})
    print(f"{os.path.basename(path)}: {len(preds)} tasks, mean coverage {sum(cov)/len(cov):.0%}, "
          f"complete rankings {full}/{len(preds)}, distinct rankings {distinct}")
    for p in preds[:show]:
        print("  ", p["task_id"], "->", p["output"][:300].replace("\n", " "))

In [ ]:
# 4. Zero-shot base model (no fine-tuning), Prompt A, all tasks
PRED_DIR = f"{WORK_DIR}/predictions"
os.makedirs(PRED_DIR, exist_ok=True)
t0 = time.time()
!python -m metapilot llm-predict --prompts {PROMPTS} --kind numeric --model {MODEL} --out {PRED_DIR}/base_numeric{SUFFIX}.jsonl
if _exit_code: raise RuntimeError("metapilot llm-predict failed (see the error above); stopping")
print(f"{(time.time() - t0) / 60:.1f} min")
check(f"{PRED_DIR}/base_numeric{SUFFIX}.jsonl")

In [ ]:
# 5. Fine-tune one adapter per fold and prompt kind, then predict its held-out fold.
# Look at the coverage printed after the first fold: if it is far below 100%, stop and investigate.
for kind in KINDS:
    for k in folds:
        adapter = f"{WORK_DIR}/adapters/{kind}_fold{k}"
        t0 = time.time()
        if not os.path.exists(f"{adapter}/train_info.json"):
            !python -m metapilot llm-train --prompts {PROMPTS} --fold {k} --kind {kind} --model {MODEL} --epochs {EPOCHS} --out {adapter}
            if _exit_code: raise RuntimeError("metapilot llm-train failed (see the error above); stopping")
        !python -m metapilot llm-predict --prompts {PROMPTS} --kind {kind} --fold {k} --model {MODEL} --adapter {adapter} --out {PRED_DIR}/ft_{kind}{SUFFIX}.jsonl
        if _exit_code: raise RuntimeError("metapilot llm-predict failed (see the error above); stopping")
        print(f"--- {kind} fold {k}: {(time.time() - t0) / 60:.1f} min")
        check(f"{PRED_DIR}/ft_{kind}{SUFFIX}.jsonl", show=1)

In [ ]:
# 6. Score all methods on the shared folds; LLM rows show llm_coverage
import shutil
import pandas as pd
os.makedirs(f"{META}/llm/predictions", exist_ok=True)
for f in os.listdir(PRED_DIR):
    shutil.copy(f"{PRED_DIR}/{f}", f"{META}/llm/predictions/{f}")
!python -m metapilot evaluate --config {CONFIG} --meta {META} --out runs/phase2
if _exit_code: raise RuntimeError("metapilot evaluate failed (see the error above); stopping")
cols = ["regret@1", "regret@1_lo", "regret@1_hi", "regret@3", "top1_hit", "spearman", "n_tasks", "llm_coverage",
        "distinct_rankings", "distinct_top1"]
s = pd.read_csv("runs/phase2/summary.csv", index_col=0)
display(s[[c for c in cols if c in s]].round(3))
r = pd.read_csv("runs/phase2/summary_real.csv", index_col=0)   # RQ3: real datasets only
display(r[[c for c in cols if c in r]].round(3))
shutil.copytree("runs/phase2", f"{WORK_DIR}/runs_phase2", dirs_exist_ok=True)
print("saved to", f"{WORK_DIR}/runs_phase2")

**What to bring back:** the work folder printed in step 2 (Colab: `MyDrive/metapilot_phase2/<RUN_NAME>/`;
Kaggle: the version's output, `metapilot_phase2/<RUN_NAME>/`). `predictions/*.jsonl`
are the raw model outputs; copy them into `<meta>/llm/predictions/` of the same meta-dataset and run
`metapilot evaluate` to reproduce the table. If an LLM row's `llm_coverage` is well below 1, the model is not
producing complete rankings and its metrics are mostly the single-best fallback; report that, not the row alone.